# Customer Churn Prediction using Deep Neural Networks (DNN)
This notebook trains a DNN on the cleaned customer churn dataset.

## 1. Import Libraries
We import libraries for data handling, preprocessing, deep learning, visualization and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,roc_curve,auc,ConfusionMatrixDisplay

## 2. Load Dataset

In [ ]:
df=pd.read_csv('../datasets/Cleaned_Customer_Churn.csv')
df.head()

## 3. Prepare Features and Target
Following the same preprocessing as the Logistic Regression notebook. The target is `Exited`.

In [ ]:
X=df.drop('Exited',axis=1)
y=df['Exited']
print(X.shape,y.shape)

## 4. Train-Test Split and Feature Scaling

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

## 5. Build the Deep Neural Network

In [ ]:
model=Sequential([
Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
BatchNormalization(),
Dropout(0.3),
Dense(32,activation='relu'),
Dropout(0.2),
Dense(16,activation='relu'),
Dense(1,activation='sigmoid')
])
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

## 6. Train the Model

In [ ]:
early=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
checkpoint=ModelCheckpoint('../models/churn_dnn.keras',save_best_only=True,monitor='val_loss')
history=model.fit(X_train,y_train,validation_split=0.2,epochs=100,batch_size=32,callbacks=[early,checkpoint],verbose=1)

## 7. Visualize Training History

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'],label='Train')
plt.plot(history.history['val_accuracy'],label='Validation')
plt.legend();plt.title('Accuracy');plt.show()
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'],label='Train')
plt.plot(history.history['val_loss'],label='Validation')
plt.legend();plt.title('Loss');plt.show()

## 8. Evaluate the Model

In [ ]:
pred_prob=model.predict(X_test)
pred=(pred_prob>0.5).astype(int)
print('Accuracy:',accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()

## 9. ROC Curve

In [ ]:
fpr,tpr,_=roc_curve(y_test,pred_prob)
roc_auc=auc(fpr,tpr)
plt.figure(figsize=(5,5))
plt.plot(fpr,tpr,label=f'AUC={roc_auc:.3f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate');plt.ylabel('True Positive Rate')
plt.legend();plt.show()

## 10. Save and Load Model

In [ ]:
model.save('../models/churn_dnn.keras')
loaded=tf.keras.models.load_model('../models/churn_dnn.keras')
print('Model saved and loaded successfully.')